# Working with JSON and SQL
Code for the Note `note.md`. Run the cells from top to bottom.

In [ ]:
# pandas: the table library
import pandas as pd

## 1. Reading a JSON file
`data/train.json` holds recipes: an id, a cuisine and a list of ingredients for each dish.

In [ ]:
# read the JSON file into a DataFrame (one row per recipe)
recipes = pd.read_json("data/train.json")
recipes.head()

In [ ]:
# rows, columns
recipes.shape

In [ ]:
# the ingredients cell of the first row is a Python list
first = recipes.loc[0, "ingredients"]
print(type(first))
print(first)

In [ ]:
# how many dishes of each cuisine: this is the label a model would predict
recipes["cuisine"].value_counts()

## 2. Reading JSON from a URL
The same function accepts a web address. This one returns today's exchange rates with the Indian rupee as the base.

In [ ]:
url = "https://api.exchangerate-api.com/v4/latest/INR"
try:
    # read straight from the web
    rates = pd.read_json(url)
except Exception as e:
    # no internet, or the address stopped working: use the copy saved in data/
    print("Could not reach the URL, using the saved copy:", e)
    rates = pd.read_json("data/exchange_rates_INR.json")
rates.head()

In [ ]:
# one row per currency, 7 columns
rates.shape

In [ ]:
# 1 rupee in a few other currencies
rates.loc[["USD", "EUR", "GBP", "JPY"], "rates"]

## 3. Reading SQL into pandas
`data/world.db` is the `world` sample database as an SQLite file (built from `world.sql` by `data/make_world_db.py`). SQLite comes with Python, so no database server is needed.

In [ ]:
# sqlite3: Python's built-in connector for SQLite databases
import sqlite3

# open a connection to the database file
conn = sqlite3.connect("data/world.db")

In [ ]:
# which tables does the database hold?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

In [ ]:
# the whole city table
city = pd.read_sql_query("SELECT * FROM city", conn)
print(city.shape)
city.head()

In [ ]:
# only the cities in India
india = pd.read_sql_query("SELECT * FROM city WHERE CountryCode = 'IND'", conn)
print(india.shape)
india.head()

In [ ]:
# only the cities in the USA
usa = pd.read_sql_query("SELECT * FROM city WHERE CountryCode = 'USA'", conn)
usa.shape

In [ ]:
# the country table, keeping only countries with life expectancy above 60
countries = pd.read_sql_query("SELECT * FROM country WHERE LifeExpectancy > 60", conn)
print(countries.shape)
countries[["Name", "Continent", "LifeExpectancy"]].head()

In [ ]:
# index_col: use the ID column as the row labels
pd.read_sql_query("SELECT * FROM city", conn, index_col="ID").head()

In [ ]:
# chunksize: read 1000 rows at a time instead of all at once
for chunk in pd.read_sql_query("SELECT * FROM city", conn, chunksize=1000):
    print(chunk.shape)

In [ ]:
# close the connection when done
conn.close()

## 4. The same with MySQL (not run here)
With the `world` database loaded into a MySQL server (for example through XAMPP), only the connection changes:

```python
# pip install mysql-connector-python
import mysql.connector

conn = mysql.connector.connect(host="localhost", user="root", password="", database="world")
df = pd.read_sql_query("SELECT * FROM countrylanguage", conn)
```

pandas prints a warning for connections other than SQLite and SQLAlchemy; it still works. The officially supported way is an SQLAlchemy engine:

```python
from sqlalchemy import create_engine

engine = create_engine("mysql+mysqlconnector://root:@localhost/world")
df = pd.read_sql_query("SELECT * FROM countrylanguage", engine)
```